# Renewable Energy Insights: Advanced SQL (CTEs & Window Functions) applied to Climate Data

An end-to-end data pipeline built with Pandas and SQL, analyzing global renewable energy trends and calculating year-over-year climate impacts

# Data Ingestion

Load the raw datasets that compose the renewable energy warehouse:

- Country Dimension
- Energy Source Dimension
- Time Dimension
- Energy Fact Table

In [ ]:
# lib import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

: 

In [ ]:
# data import
df = pd.read_csv('../raw/dim_country.csv')
df2 = pd.read_csv('../raw/dim_energy_source.csv')
df3 = pd.read_csv('../raw/dim_time.csv')
df4 = pd.read_csv('../raw/fact_energy.csv')

# Data Quality Assessment

Before transformations, the dataset is inspected for:

- Missing values
- Duplicated records
- Referential integrity issues
- Invalid measurements

In [ ]:
df4.info()
df4.head()

# Missing/Duplicated values in fact table
print("Missing values in fact table:")
print(df4.isna().sum())
print("\nDuplicated values in fact table")
print(df4.duplicated().sum())

In [ ]:
# Now we check types on the dim table to ensure we're working with the correct types
print("Check on the column types: ")
print(df3.dtypes)

# Example: We could force a date column to be an actual datetime object
# df3['date_column'] = pd.to_datetime(df3['date_column'])

In [ ]:
# Check if all country IDs in the fact table exist in the dimension table
orphaned_countries = ~df4['country_id'].isin(df['country_id'])
print(f"Orphaned records found: {orphaned_countries.sum()}")

In [ ]:
df4.info()
# Now we give a closer look on the data.
print(df4['value'].describe())

# Hunting negative values
print(f"\nNegative values: {(df4['value'] < 0).sum()}")

# And possible duplicates
print(f"Duplicate rows: {df4.duplicated(subset=['country_id', 'time_id', 'source_id']).sum()}")


# Data Cleaning

This section performs a few checks removes duplicate records and prepares the dataset for analysis.

In [ ]:
# Missing values were identified but not removed.
# In a production scenario the decision would depend on business rules.
df4 = df4.dropna()

In [ ]:
# 1. Isolate the duplicates and sort them so they appear right next to each other
duplicates_preview = df4[df4.duplicated(subset=['country_id', 'time_id', 'source_id'], keep=False)]

# 2. Look at the first 10 rows to see what is happening
print(duplicates_preview.sort_values(by=['country_id', 'time_id', 'source_id']).head(10))

### Duplicate Analysis

The fact table should contain only one measurement for each:

(country_id, time_id, source_id)

Any repeated combination indicates a data quality issue.

In [ ]:
# Drop duplicates keeping only the first occurrence
df4_clean = df4.drop_duplicates(subset=['country_id', 'time_id', 'source_id'], keep='first')

# Verify the fix
print(f"Old row count: {len(df4)}")
print(f"New row count: {len(df4_clean)}")
removed = len(df4) - len(df4_clean)
print(f"Removed rows: {removed}")
print(f"Reduction: {(removed/len(df4))*100:.2f}%")

print(f"Remaining duplicates: {df4_clean.duplicated(subset=['country_id', 'time_id', 'source_id']).sum()}")

In [ ]:

df_clean = df4_clean.merge(df, on='country_id', how='inner')
df_clean = df_clean.merge(df2, on='source_id', how='inner')
df_clean = df_clean.merge(df3, on='time_id', how='inner')
df_clean = df_clean.drop(columns=['country_id', 'time_id', 'source_id', 'fact_id'])

df_clean.to_csv('../processed/clean_renewable_energy.csv', index=False)
print("Pipeline Extraction and Transformation Complete. CSV exported.")

# Data Warehouse Assembly

Dimension tables are merged with the fact table to create a human-readable analytical dataset

In [ ]:
# Quick merge to see results
df_preview = df4.merge(df, on='country_id', how='inner')
df_preview = df_preview.merge(df2, on='source_id', how='inner')

# Let's look at the first 5 rows of the combined, human-readable data
print(df_preview.head())

# SQL Analytics

Business questions are answered using SQL executed through SQLite.

In [ ]:
conn = sqlite3.connect(':memory:')

# Load DataFrame into the SQL engine as a table named 'energy'
df_clean.to_sql('energy', conn, index=False)

query = """
SELECT * FROM energy 
LIMIT 5;
"""
print(pd.read_sql(query, conn))

## Analytical Question 1

Which countries have historically maintained more than 50% renewable participation in their energy matrix?

In [ ]:
query_cte = """
WITH MediaHistorica AS (
    SELECT 
        entity AS pais, 
        AVG(value) AS media_renovavel
    FROM energy
    WHERE source_name = 'renewables_pct_primary_energy'
	AND entity NOT IN ('World', 'Africa', 'Europe', 'South America', 'Asia')
    GROUP BY entity
)

SELECT 
    pais, 
    media_renovavel
FROM MediaHistorica
WHERE media_renovavel > 50
ORDER BY media_renovavel DESC
LIMIT 10;
"""

# Executando a query e mostrando o resultado no Pandas
print(pd.read_sql(query_cte, conn)) 

## Analytical Question 2

How has Brazil's renewable share evolved over time and what was its year-over-year growth?

In [ ]:
query_window = """
WITH HistoricoBrasil AS (
    SELECT 
        year AS ano,
        value AS renovavel_pct
    FROM energy
    WHERE entity = 'Brazil' 
      AND source_name = 'renewables_pct_primary_energy'
),

BrasilComLag AS (
    SELECT 
        ano,
        renovavel_pct,
        LAG(renovavel_pct) OVER (ORDER BY ano ASC) AS ano_anterior
    FROM HistoricoBrasil
)

SELECT 
    ano,
    ROUND(renovavel_pct, 2) AS pct_atual,
    ROUND(ano_anterior, 2) AS pct_ano_passado,
    ROUND(renovavel_pct - ano_anterior, 2) AS crescimento_pontos_percentuais
FROM BrasilComLag
WHERE ano_anterior IS NOT NULL 
ORDER BY ano DESC
LIMIT 10;
"""

print(pd.read_sql(query_window, conn))

In [ ]:
df_trends = pd.read_sql(query_window, conn)

sns.set_theme(style="darkgrid")
fig, ax1 = plt.subplots(figsize=(12,6))

sns.lineplot(data=df_trends, x='ano', y='pct_atual', marker='o', linewidth=2.5, color='purple', ax=ax1, label='% Renewable')
ax1.set_ylabel('Renewable Share(%)', fontsize=12)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_title('Brazil: Renewable Energy Share & Year-over-Year Growth (2012-2021)', fontsize=16, pad=20)

ax2 = ax1.twinx()

colors = ['crimson' if val < 0 else 'mediumseagreen' for val in df_trends['crescimento_pontos_percentuais']]
sns.barplot(data=df_trends, x='ano', y='crescimento_pontos_percentuais', hue='ano', palette=colors, alpha=0.6, ax=ax2, legend=False)
ax2.set_ylabel('YoY Growth (Percentage Points)', fontsize=12)

plt.show()
plt.tight_layout()
plt.savefig(
    "finished/brazil_renewable_trend.png",
    dpi=300,
    bbox_inches="tight"
)